# Acertos por proximidade de percentual (tse3)

Usa os `votos_mun_{ANO}.csv` já gerados (não precisa escolher ano nem rodar passos 0–2).

**Critério de acerto em cada turno** (ACERTO1 / ACERTO2):
1. O município tem **vencedor local único** (sem empate) e é o **mesmo** do vencedor nacional.
2. O **percentual de votos** desse vencedor no município está a no máximo `TOLERANCIA_PCT` pontos percentuais (pp) do percentual nacional (para mais ou para menos).

**Fluxo:**
1. Ajuste `TOLERANCIA_PCT` na config e rode a célula.
2. Rode a célula que **gera** `acertos_mun_{ANO}.csv` para todos os anos.
3. Rode o **cruzamento multi-ano** (mesma ideia do tse.ipynb).

Para testar vários limiares: mude o percentual, rode de novo a geração e depois o cruzamento.

In [7]:
# Configuração
import re
import pandas as pd
from pathlib import Path

# Tolerância em pontos percentuais (pp): ex. 10 → vencedor com 52% no Brasil
# aceita no município entre 42% e 62%
TOLERANCIA_PCT = 7.0

PASTA = Path(".")
ENCODING = "latin-1"
SEPARADOR = ";"
VOTOS_INVALIDOS = {"95", "96", "97"}

CHAVE_CRUZAMENTO = "CD_MUNICIPIO"
CHAVES_CIDADE = [CHAVE_CRUZAMENTO]
COLUNAS_ACERTOS_SAIDA = ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO", "ACERTO1", "ACERTO2"]


def referencia_municipios(df):
    return (
        df[["CD_MUNICIPIO", "SG_UF", "NM_MUNICIPIO"]]
        .drop_duplicates(subset=["CD_MUNICIPIO"], keep="first")
    )


def normalizar_cd_municipio(serie):
    return serie.astype(str).str.strip().str.lstrip("0").replace("", "0")


def listar_votos_mun(pasta=PASTA):
    arquivos = sorted(
        pasta.glob("votos_mun_*.csv"),
        key=lambda p: int(p.stem.split("_")[-1]),
    )
    if not arquivos:
        raise FileNotFoundError(f"Nenhum votos_mun_*.csv em {pasta.resolve()}")
    return arquivos


print("--- Configuração tse3 ---")
print(f"Tolerância: ±{TOLERANCIA_PCT} pp (pontos percentuais)")
print(f"Pasta: {PASTA.resolve()}")
try:
    anos_disp = [int(p.stem.split("_")[-1]) for p in listar_votos_mun()]
    print(f"votos_mun encontrados: {len(anos_disp)} anos → {anos_disp}")
except FileNotFoundError as e:
    print(e)

--- Configuração tse3 ---
Tolerância: ±7.0 pp (pontos percentuais)
Pasta: D:\cursos\python\estatisticas\tse
votos_mun encontrados: 8 anos → [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]


In [8]:
# Gera acertos_mun_{ANO}.csv para todos os votos_mun_*.csv


def percentuais_turno(votos_mun, turno):
    validos = votos_mun[
        (votos_mun["NR_TURNO"] == turno)
        & (~votos_mun["NR_VOTAVEL"].isin(VOTOS_INVALIDOS))
    ]
    totais = validos.groupby("NR_VOTAVEL")["QT_VOTOS"].sum()
    if totais.empty or totais.sum() == 0:
        return None, None
    pct = totais / totais.sum() * 100
    return totais.idxmax(), pct


def acertos_por_turno_pct(votos_mun, turno, tolerancia_pp):
    """Acerto = vencedor local único = nacional e % do vencedor dentro da tolerância."""
    turno_df = votos_mun[votos_mun["NR_TURNO"] == turno]
    validos = turno_df[~turno_df["NR_VOTAVEL"].isin(VOTOS_INVALIDOS)].copy()
    resultado = validos[CHAVES_CIDADE].drop_duplicates()

    venc_nacional, pct_nac = percentuais_turno(votos_mun, turno)
    if venc_nacional is None:
        resultado["acerto"] = 0
        return resultado, None, None

    pct_nac_venc = float(pct_nac[venc_nacional])

    tot_mun = validos.groupby(CHAVE_CRUZAMENTO)["QT_VOTOS"].sum()
    votos_venc_mun = (
        validos[validos["NR_VOTAVEL"] == venc_nacional]
        .groupby(CHAVE_CRUZAMENTO)["QT_VOTOS"]
        .sum()
    )
    pct_local_venc = votos_venc_mun.reindex(tot_mun.index, fill_value=0) / tot_mun * 100
    dentro_pct = (pct_local_venc - pct_nac_venc).abs() <= tolerancia_pp

    max_v = validos.groupby(CHAVE_CRUZAMENTO)["QT_VOTOS"].transform("max")
    lideres = validos[validos["QT_VOTOS"] == max_v]
    unico = lideres.groupby(CHAVE_CRUZAMENTO).size() == 1
    venc_local = lideres.drop_duplicates(CHAVE_CRUZAMENTO).set_index(CHAVE_CRUZAMENTO)[
        "NR_VOTAVEL"
    ]
    acertou_vencedor = unico & (venc_local == venc_nacional)

    idx = resultado[CHAVE_CRUZAMENTO]
    acerto = (
        acertou_vencedor.reindex(idx, fill_value=False).values
        & dentro_pct.reindex(idx, fill_value=False).values
    ).astype(int)
    resultado["acerto"] = acerto
    return resultado, venc_nacional, pct_nac_venc


def gerar_acertos_municipio(
    arquivo_entrada,
    arquivo_saida,
    tolerancia_pp=TOLERANCIA_PCT,
    forcar=True,
):
    if not arquivo_entrada.exists():
        raise FileNotFoundError(arquivo_entrada)

    votos = pd.read_csv(arquivo_entrada, sep=SEPARADOR, encoding=ENCODING, dtype=str)
    votos["QT_VOTOS"] = pd.to_numeric(votos["QT_VOTOS"], errors="coerce").fillna(0)

    ref = referencia_municipios(votos)
    t1, v1, pct1 = acertos_por_turno_pct(votos, "1", tolerancia_pp)
    t1 = t1.rename(columns={"acerto": "ACERTO1"})
    tem_turno2 = (votos["NR_TURNO"] == "2").any()

    if tem_turno2:
        t2, v2, pct2 = acertos_por_turno_pct(votos, "2", tolerancia_pp)
        t2 = t2.rename(columns={"acerto": "ACERTO2"})
        acertos = t1.merge(t2, on=CHAVES_CIDADE, how="inner")
    else:
        acertos = t1.copy()
        acertos["ACERTO2"] = ""
        v2, pct2 = None, None

    acertos = ref.merge(acertos, on=CHAVE_CRUZAMENTO, how="right")[COLUNAS_ACERTOS_SAIDA]
    acertos.to_csv(arquivo_saida, sep=SEPARADOR, encoding=ENCODING, index=False)

    ano = int(arquivo_entrada.stem.split("_")[-1])
    n1 = int(acertos["ACERTO1"].astype(int).sum())
    linha = f"  {ano}: venc. 1º={v1} ({pct1:.2f}% BR) → {n1:,} municípios (±{tolerancia_pp} pp)"
    if tem_turno2:
        n2 = int(acertos["ACERTO2"].astype(int).sum())
        linha += f" | 2º={v2} ({pct2:.2f}% BR) → {n2:,} municípios"
    else:
        linha += " | sem 2º turno (ACERTO2 vazio)"
    print(linha)
    return acertos


def gerar_acertos_todos_anos(
    pasta=PASTA,
    tolerancia_pp=TOLERANCIA_PCT,
    forcar=True,
):
    arquivos = listar_votos_mun(pasta)
    print(
        f"Gerando acertos_mun_*.csv para {len(arquivos)} anos "
        f"(tolerância ±{tolerancia_pp} pp)...\n"
    )
    for arq in arquivos:
        ano = int(arq.stem.split("_")[-1])
        saida = pasta / f"acertos_mun_{ano}.csv"
        gerar_acertos_municipio(arq, saida, tolerancia_pp=tolerancia_pp, forcar=forcar)
    print("\nConcluído.")


gerar_acertos_todos_anos()

Gerando acertos_mun_*.csv para 8 anos (tolerância ±7.0 pp)...

  1994: venc. 1º=45 (54.28% BR) → 1,154 municípios (±7.0 pp) | sem 2º turno (ACERTO2 vazio)
  1998: venc. 1º=45 (53.06% BR) → 1,449 municípios (±7.0 pp) | sem 2º turno (ACERTO2 vazio)
  2002: venc. 1º=13 (46.44% BR) → 2,197 municípios (±7.0 pp) | 2º=13 (61.27% BR) → 1,954 municípios
  2006: venc. 1º=13 (48.60% BR) → 817 municípios (±7.0 pp) | 2º=13 (60.83% BR) → 1,382 municípios
  2010: venc. 1º=13 (46.91% BR) → 1,326 municípios (±7.0 pp) | 2º=13 (56.05% BR) → 1,726 municípios
  2014: venc. 1º=13 (41.59% BR) → 581 municípios (±7.0 pp) | 2º=13 (51.64% BR) → 896 municípios
  2018: venc. 1º=17 (46.03% BR) → 1,268 municípios (±7.0 pp) | 2º=17 (55.13% BR) → 1,061 municípios
  2022: venc. 1º=13 (48.43% BR) → 878 municípios (±7.0 pp) | 2º=13 (50.90% BR) → 686 municípios

Concluído.


In [9]:
# Cruzamento entre anos (só CD_MUNICIPIO)


def df_tem_acerto2(df):
    if "ACERTO2" not in df.columns:
        return False
    return df["ACERTO2"].fillna("").astype(str).str.strip().ne("").any()


def referencia_nomes_acertos(arquivos):
    for arq in reversed(arquivos):
        df = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, dtype=str)
        if {"SG_UF", "NM_MUNICIPIO"}.issubset(df.columns):
            return referencia_municipios(df)
    return None


def carregar_acertos_existentes(pasta=PASTA):
    arquivos = sorted(
        pasta.glob("acertos_mun_*.csv"),
        key=lambda p: int(p.stem.split("_")[-1]),
    )
    if not arquivos:
        raise FileNotFoundError(f"Nenhum acertos_mun_*.csv em {pasta.resolve()}")

    combinado = None
    anos = []

    for arq in arquivos:
        ano = int(arq.stem.split("_")[-1])
        anos.append(ano)
        df = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, dtype=str)
        df[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(df[CHAVE_CRUZAMENTO])
        df = df.rename(columns={"ACERTO1": f"ACERTO1_{ano}"})
        df[f"ACERTO1_{ano}"] = df[f"ACERTO1_{ano}"].astype(int)
        cols_ano = [CHAVE_CRUZAMENTO, f"ACERTO1_{ano}"]

        if df_tem_acerto2(df):
            df = df.rename(columns={"ACERTO2": f"ACERTO2_{ano}"})
            df[f"ACERTO2_{ano}"] = pd.to_numeric(df[f"ACERTO2_{ano}"], errors="coerce").astype(int)
            cols_ano.append(f"ACERTO2_{ano}")
        else:
            print(f"  {ano}: só 1º turno — ACERTO2 não entra no cruzamento")

        df = df[cols_ano]
        combinado = df if combinado is None else combinado.merge(df, on=CHAVE_CRUZAMENTO, how="inner")

    ref = referencia_nomes_acertos(arquivos)
    if ref is not None:
        combinado = ref.merge(combinado, on=CHAVE_CRUZAMENTO, how="right")

    colunas_acerto = [c for c in combinado.columns if c.startswith("ACERTO")]
    combinado["acertou_tudo"] = combinado[colunas_acerto].eq(1).all(axis=1)
    combinado["qtd_acertos"] = combinado[colunas_acerto].sum(axis=1)
    combinado["qtd_possivel"] = len(colunas_acerto)
    n_t1 = len([c for c in colunas_acerto if re.fullmatch(r"ACERTO1_\d{4}", c)])
    n_t2 = len([c for c in colunas_acerto if re.fullmatch(r"ACERTO2_\d{4}", c)])
    print(
        f"Critérios de acerto: {len(colunas_acerto)} "
        f"({n_t1}× 1º turno, {n_t2}× 2º turno) | tolerância usada na geração: ±{TOLERANCIA_PCT} pp"
    )
    return combinado, anos, arquivos


acertos_todos, anos_disponiveis, arquivos_acertos = carregar_acertos_existentes()

print("Arquivos de acertos:")
for arq in arquivos_acertos:
    print(f"  - {arq.name}")
print(f"\nAnos: {anos_disponiveis}")

perfeitas = acertos_todos[acertos_todos["acertou_tudo"]].copy()
print(f"\nMunicípios que acertaram tudo ({len(perfeitas):,}):")
if perfeitas.empty:
    print("  Nenhum — tente aumentar TOLERANCIA_PCT e regerar os acertos.")
else:
    cols_exibir = ["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"] + [
        c for c in perfeitas.columns if c.startswith("ACERTO")
    ] + ["qtd_acertos"]
    cols_exibir = [c for c in cols_exibir if c in perfeitas.columns]
    display(perfeitas.sort_values(["SG_UF", "NM_MUNICIPIO"])[cols_exibir])

    ARQUIVO_PERFEITAS = PASTA / f"municipios_acertaram_tudo_tol{int(TOLERANCIA_PCT)}pp.csv"
    lista = (
        perfeitas[["SG_UF", "CD_MUNICIPIO", "NM_MUNICIPIO"]]
        .drop_duplicates(subset=["CD_MUNICIPIO"])
        .sort_values(["SG_UF", "NM_MUNICIPIO"])
    )
    lista.to_csv(ARQUIVO_PERFEITAS, sep=SEPARADOR, encoding=ENCODING, index=False)
    print(f"\nLista salva: {ARQUIVO_PERFEITAS.resolve()} ({len(lista):,} municípios)")

quase = acertos_todos[acertos_todos["qtd_acertos"] == acertos_todos["qtd_possivel"] - 1]
print(f"\nMunicípios com 1 erro apenas: {len(quase):,}")
display(quase.head(10))

  1994: só 1º turno — ACERTO2 não entra no cruzamento
  1998: só 1º turno — ACERTO2 não entra no cruzamento
Critérios de acerto: 14 (8× 1º turno, 6× 2º turno) | tolerância usada na geração: ±7.0 pp
Arquivos de acertos:
  - acertos_mun_1994.csv
  - acertos_mun_1998.csv
  - acertos_mun_2002.csv
  - acertos_mun_2006.csv
  - acertos_mun_2010.csv
  - acertos_mun_2014.csv
  - acertos_mun_2018.csv
  - acertos_mun_2022.csv

Anos: [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

Municípios que acertaram tudo (1):


,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,ACERTO1_1994,ACERTO1_1998,ACERTO1_2002,ACERTO2_2002,ACERTO1_2006,ACERTO2_2006,ACERTO1_2010,ACERTO2_2010,ACERTO1_2014,ACERTO2_2014,ACERTO1_2018,ACERTO2_2018,ACERTO1_2022,ACERTO2_2022,qtd_acertos
2551,RJ,58238,CARMO,1,1,1,1,1,1,1,1,1,1,1,1,1,1,14



Lista salva: D:\cursos\python\estatisticas\tse\municipios_acertaram_tudo_tol7pp.csv (1 municípios)

Municípios com 1 erro apenas: 3


,CD_MUNICIPIO,SG_UF,NM_MUNICIPIO,ACERTO1_1994,ACERTO1_1998,ACERTO1_2002,ACERTO2_2002,ACERTO1_2006,ACERTO2_2006,ACERTO1_2010,ACERTO2_2010,ACERTO1_2014,ACERTO2_2014,ACERTO1_2018,ACERTO2_2018,ACERTO1_2022,ACERTO2_2022,acertou_tudo,qtd_acertos,qtd_possivel
2385,53678,MG,TARUMIRIM,1,1,1,1,1,1,1,1,1,0,1,1,1,1,False,13,14
2723,61883,SP,SÃO JOÃO DE IRACEMA,0,1,1,1,1,1,1,1,1,1,1,1,1,1,False,13,14
4541,90638,MS,CORUMBÁ,1,1,1,1,1,1,1,1,1,1,1,1,1,0,False,13,14


In [10]:
# Comparativo: vencedor nacional × % no município (todos os anos)
CD_MUNICIPIO_ANALISE = "58238"
ARQUIVO_COMPARATIVO_MUN = PASTA / f"comparativo_vencedor_mun_{CD_MUNICIPIO_ANALISE}.csv"

PARTIDO_VENCEDOR_FALLBACK = {
    (1994, "1", "45"): "PSDB",
    (1998, "1", "45"): "PSDB",
    (2002, "1", "13"): "PT",
    (2002, "2", "13"): "PT",
    (2006, "1", "13"): "PT",
    (2006, "2", "13"): "PT",
    (2010, "1", "13"): "PT",
    (2010, "2", "13"): "PT",
    (2014, "1", "13"): "PT",
    (2014, "2", "13"): "PT",
    (2018, "1", "17"): "PSL",
    (2018, "2", "17"): "PSL",
    (2022, "1", "13"): "PT",
    (2022, "2", "13"): "PT",
}


def candidatos_arquivo_original(ano):
    return [
        PASTA / f"votacao_secao_{ano}_BR.csv",
        PASTA / f"votacao_candidato_munzona_{ano}_BRASIL.csv",
        PASTA / f"votacao_candidato_munzona_{ano}_BR.csv",
    ]


def mapa_partidos_ano(ano):
    """NR_VOTAVEL -> SG_PARTIDO a partir do CSV original do TSE (se existir na pasta)."""
    mapa = {}
    for arq in candidatos_arquivo_original(ano):
        if not arq.exists():
            continue
        cols = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, nrows=0).columns
        col_partido = next(
            (c for c in cols if c in ("SG_PARTIDO", "SIGLA_PARTIDO", "SG_PARTIDO_CANDIDATO")),
            None,
        )
        if not col_partido:
            continue
        col_nr = "NR_VOTAVEL" if "NR_VOTAVEL" in cols else "NR_CANDIDATO"
        for chunk in pd.read_csv(
            arq,
            sep=SEPARADOR,
            encoding=ENCODING,
            usecols=[col_nr, col_partido],
            dtype=str,
            chunksize=500_000,
        ):
            pares = chunk.drop_duplicates(subset=[col_nr])
            for _, row in pares.iterrows():
                nr = str(row[col_nr]).strip()
                partido = str(row[col_partido]).strip()
                if nr and partido and nr not in mapa:
                    mapa[nr] = partido
        if mapa:
            break
    return mapa


def partido_vencedor(ano, turno, nr_votavel, mapa_ano):
    nr = str(nr_votavel)
    if nr in mapa_ano:
        return mapa_ano[nr]
    return PARTIDO_VENCEDOR_FALLBACK.get((ano, str(turno), nr), "")


def pct_vencedor_no_municipio(votos_mun, turno, cd_municipio, nr_vencedor):
    cd = str(cd_municipio)
    turno_df = votos_mun[
        (votos_mun["NR_TURNO"] == str(turno))
        & (votos_mun[CHAVE_CRUZAMENTO] == cd)
        & (~votos_mun["NR_VOTAVEL"].isin(VOTOS_INVALIDOS))
    ]
    if turno_df.empty:
        return None
    total = turno_df["QT_VOTOS"].sum()
    if total == 0:
        return None
    votos_venc = turno_df.loc[turno_df["NR_VOTAVEL"] == str(nr_vencedor), "QT_VOTOS"].sum()
    return float(votos_venc / total * 100)


def nome_vencedor(votos_mun, turno, nr_vencedor):
    linhas = votos_mun[
        (votos_mun["NR_TURNO"] == str(turno))
        & (votos_mun["NR_VOTAVEL"] == str(nr_vencedor))
    ]["NM_VOTAVEL"]
    return linhas.iloc[0] if not linhas.empty else ""


def gerar_comparativo_municipio(cd_municipio=CD_MUNICIPIO_ANALISE):
    linhas = []
    for arq in listar_votos_mun():
        ano = int(arq.stem.split("_")[-1])
        votos = pd.read_csv(arq, sep=SEPARADOR, encoding=ENCODING, dtype=str)
        votos[CHAVE_CRUZAMENTO] = normalizar_cd_municipio(votos[CHAVE_CRUZAMENTO])
        votos["QT_VOTOS"] = pd.to_numeric(votos["QT_VOTOS"], errors="coerce").fillna(0)
        mapa_part = mapa_partidos_ano(ano)

        if cd_municipio not in set(votos[CHAVE_CRUZAMENTO]):
            print(f"  {ano}: município {cd_municipio} não encontrado em {arq.name}")
            continue

        nm_mun = (
            votos.loc[votos[CHAVE_CRUZAMENTO] == str(cd_municipio), "NM_MUNICIPIO"]
            .iloc[0]
        )

        for turno in sorted(votos["NR_TURNO"].dropna().unique(), key=lambda x: int(x)):
            venc, pct_nac = percentuais_turno(votos, turno)
            if venc is None:
                continue
            pct_br = round(float(pct_nac[venc]), 2)
            pct_mun = pct_vencedor_no_municipio(votos, turno, cd_municipio, venc)
            if pct_mun is None:
                continue
            linhas.append(
                {
                    "ano": ano,
                    "turno": int(turno),
                    "partido do vencedor": partido_vencedor(
                        ano, turno, venc, mapa_part
                    ),
                    "nome do vencedor": nome_vencedor(votos, turno, venc),
                    "% de votos do vencedor no Brasil": pct_br,
                    f"% de votos do vencedor no município {cd_municipio}": round(
                        pct_mun, 2
                    ),
                }
            )

    if not linhas:
        raise ValueError(f"Nenhuma linha gerada para CD_MUNICIPIO={cd_municipio}")

    comparativo = pd.DataFrame(linhas)
    comparativo.to_csv(
        ARQUIVO_COMPARATIVO_MUN, sep=SEPARADOR, encoding=ENCODING, index=False
    )
    print(
        f"Comparativo — {nm_mun} ({cd_municipio}), "
        f"{len(comparativo)} linhas (ano × turno)"
    )
    print(f"Arquivo: {ARQUIVO_COMPARATIVO_MUN.resolve()}")
    display(comparativo)
    return comparativo


comparativo_mun = gerar_comparativo_municipio()

Comparativo — CARMO (58238), 14 linhas (ano × turno)
Arquivo: D:\cursos\python\estatisticas\tse\comparativo_vencedor_mun_58238.csv


,ano,turno,partido do vencedor,nome do vencedor,% de votos do vencedor no Brasil,% de votos do vencedor no município 58238
0,1994,1,PSDB,FERNANDO HENRIQUE CARDOSO,54.28,60.16
1,1998,1,PSDB,FERNANDO HENRIQUE,53.06,57.44
2,2002,1,PT,LUIZ INACIO LULA DA SILVA,46.44,40.34
3,2002,2,PT,LUIZ INACIO LULA DA SILVA,61.27,64.21
4,2006,1,PT,LUIZ INACIO LULA DA SILVA,48.60,52.45
5,2006,2,PT,LUIZ INACIO LULA DA SILVA,60.83,65.76
6,2010,1,PT,DILMA VANA ROUSSEFF,46.91,53.71
7,2010,2,PT,DILMA VANA ROUSSEFF,56.05,61.60
8,2014,1,PT,DILMA VANA ROUSSEFF,41.59,47.91
9,2014,2,PT,DILMA VANA ROUSSEFF,51.64,58.35
